# Teams history generator

Load every match JSON file, validate its structure and columns, merge all seasons, and store the result in `data/csv/teams_history.csv`.

In [316]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

JSON_DIR = Path("../data/online_sources/teams_matches_history_23-24_25-26/")
OUTPUT_PATH = Path("../data/csv/notebooks_generated/serie_a_teams_history.csv")

## Discover and validate JSON files

In [317]:
json_files = sorted(JSON_DIR.glob("*.json"))
if not json_files:
    raise FileNotFoundError(f"No JSON files found in {JSON_DIR.resolve()}")

pd.DataFrame({"json_file": [path.name for path in json_files]})

,json_file
0,matches23-24.json
1,matches24-25.json
2,matches25-26.json


In [318]:
payloads = {}
validation_errors = []

for json_path in json_files:
    try:
        with json_path.open(encoding="utf-8") as json_file:
            payload = json.load(json_file)

        if not isinstance(payload, dict):
            raise TypeError("The root element must be an object")
        if not isinstance(payload.get("matches"), list):
            raise TypeError("The 'matches' field must be a list")

        payloads[json_path.name] = payload
    except (json.JSONDecodeError, OSError, TypeError) as error:
        validation_errors.append({"json_file": json_path.name, "error": str(error)})

if validation_errors:
    display(pd.DataFrame(validation_errors))
    raise ValueError("Invalid JSON files found. Fix them before generating teams_history.csv.")

print(f"Validated {len(payloads)} JSON files.")

Validated 3 JSON files.


## Convert every JSON file to a DataFrame

In [319]:
dataframes: dict[str, pd.DataFrame] = {}

for json_file_name, payload in payloads.items():
    matches_df = pd.json_normalize(payload["matches"], sep="_")
    matches_df.insert(0, "competition", payload.get("name", ""))
    matches_df.insert(1, "source_file", json_file_name)
    dataframes[json_file_name] = matches_df

pd.DataFrame(
    {
        "json_file": file_name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
    }
    for file_name, dataframe in dataframes.items()
)

,json_file,rows,columns
0,matches23-24.json,380,9
1,matches24-25.json,380,9
2,matches25-26.json,380,10


## Check column correspondence

In [320]:
for file_name, df in dataframes.items():
    print(f"{file_name} cols: {df.columns}")

matches23-24.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ht', 'score_ft'],
      dtype='str')
matches24-25.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score_ht', 'score_ft'],
      dtype='str')
matches25-26.json cols: Index(['competition', 'source_file', 'round', 'date', 'time', 'team1', 'team2',
       'score', 'score_ft', 'score_ht'],
      dtype='str')


In [321]:
df = dataframes["matches25-26.json"].copy()
df = df[df["score"].isna()]
(df[df["score"].isna()].shape[0], df.shape[0])

(344, 344)

In [322]:
for file_name in dataframes.keys():
    for col in ["score", "source_file"]:
        if col not in dataframes[file_name].columns:
            continue
        dataframes[file_name].drop(columns=[col], inplace=True)
    print(f"{file_name} cols: {dataframes[file_name].columns}")

matches23-24.json cols: Index(['competition', 'round', 'date', 'time', 'team1', 'team2', 'score_ht',
       'score_ft'],
      dtype='str')
matches24-25.json cols: Index(['competition', 'round', 'date', 'time', 'team1', 'team2', 'score_ht',
       'score_ft'],
      dtype='str')
matches25-26.json cols: Index(['competition', 'round', 'date', 'time', 'team1', 'team2', 'score_ft',
       'score_ht'],
      dtype='str')


In [323]:
for file_name, df in dataframes.items():
    dataframes[file_name] = df.sort_index(axis=1, ascending=True)
    print(f"{file_name} cols: {dataframes[file_name].columns}")

matches23-24.json cols: Index(['competition', 'date', 'round', 'score_ft', 'score_ht', 'team1',
       'team2', 'time'],
      dtype='str')
matches24-25.json cols: Index(['competition', 'date', 'round', 'score_ft', 'score_ht', 'team1',
       'team2', 'time'],
      dtype='str')
matches25-26.json cols: Index(['competition', 'date', 'round', 'score_ft', 'score_ht', 'team1',
       'team2', 'time'],
      dtype='str')


In [324]:
reference_file = next(iter(dataframes))
reference_columns = dataframes[reference_file].columns.tolist()
reference_column_set = set(reference_columns)

column_checks = []
for file_name, dataframe in dataframes.items():
    current_columns = dataframe.columns.tolist()
    current_column_set = set(current_columns)
    column_checks.append(
        {
            "json_file": file_name,
            "same_columns": current_column_set == reference_column_set,
            "same_order": current_columns == reference_columns,
            "missing_columns": sorted(reference_column_set - current_column_set),
            "extra_columns": sorted(current_column_set - reference_column_set),
        }
    )

column_check_df = pd.DataFrame(column_checks)
display(column_check_df)

,json_file,same_columns,same_order,missing_columns,extra_columns
0,matches23-24.json,True,True,[],[]
1,matches24-25.json,True,True,[],[]
2,matches25-26.json,True,True,[],[]


## Merge and inspect the complete history

In [325]:
if not column_check_df[["same_columns", "same_order"]].all(axis=None):
    raise ValueError("The JSON DataFrames do not have matching ordered columns.")

teams_history = pd.concat(dataframes.values(), ignore_index=True)
teams_history = teams_history.sort_values(["date", "time", "team1", "team2"]).reset_index(drop=True)

print(f"Rows: {len(teams_history):,}")
print(f"Columns: {len(teams_history.columns)}")
display(teams_history.head())
display(teams_history.tail())

Rows: 1,140
Columns: 8


,competition,date,round,score_ft,score_ht,team1,team2,time
0,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[0, 1]","[0, 0]",Empoli FC,Hellas Verona FC,18:30
1,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[1, 3]","[1, 2]",Frosinone Calcio,SSC Napoli,18:30
2,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[2, 0]","[1, 0]",FC Internazionale Milano,AC Monza,20:45
3,Italian Serie A 2023/24,2023-08-19,Matchday 1,"[1, 4]","[0, 3]",Genoa CFC,ACF Fiorentina,20:45
4,Italian Serie A 2023/24,2023-08-20,Matchday 1,"[2, 2]","[1, 1]",AS Roma,US Salernitana 1919,18:30


,competition,date,round,score_ft,score_ht,team1,team2,time
1135,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[1, 2]","[1, 1]",AC Milan,Cagliari Calcio,20:45
1136,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[0, 2]","[0, 0]",Hellas Verona FC,AS Roma,20:45
1137,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[1, 4]","[0, 1]",US Cremonese,Como 1907,20:45
1138,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[1, 0]","[1, 0]",US Lecce,Genoa CFC,20:45
1139,Italian Serie A 2025/26,2026-05-24,Matchday 38,"[2, 2]","[0, 1]",Torino FC,Juventus FC,21:45


# Build the history wins, loss, goals done and goals against DataFrame

In [326]:
teams_history: pd.DataFrame = pd.DataFrame()
team_match_rows = []

for payload in payloads.values():
    season = payload["name"].rsplit(maxsplit=1)[-1].replace("/", "-")

    for match in payload["matches"]:
        score = match.get("score")
        if isinstance(score, dict):
            full_time_score = score.get("ft")
            half_time_score = score.get("ht")
        else:
            full_time_score = score
            half_time_score = [0, 0] if score == [0, 0] else None

        if not isinstance(full_time_score, list) or len(full_time_score) != 2:
            continue

        if half_time_score is None and full_time_score == [0, 0]:
            half_time_score = [0, 0]
        if not isinstance(half_time_score, list) or len(half_time_score) != 2:
            raise ValueError(f"Missing half-time score for match: {match}")

        home_goals, away_goals = map(int, full_time_score)
        home_goals_fh, away_goals_fh = map(int, half_time_score)
        home_goals_sh = home_goals - home_goals_fh
        away_goals_sh = away_goals - away_goals_fh

        team_results = [
            (match["team1"], home_goals, away_goals, home_goals_fh, home_goals_sh, away_goals_fh, away_goals_sh),
            (match["team2"], away_goals, home_goals, away_goals_fh, away_goals_sh, home_goals_fh, home_goals_sh),
        ]

        for team, goals, goals_against, goals_fh, goals_sh, goals_against_fh, goals_against_sh in team_results:
            wins = int(goals > goals_against)
            loss = int(goals < goals_against)
            draws = int(goals == goals_against)
            
            team_match_rows.append(
                {
                    "team": team,
                    "season": season,
                    "date": match["date"],
                    "time": match.get("time", ""),
                    "wins": wins,
                    "loss": int(goals < goals_against),
                    "draws": int(goals == goals_against),
                    "points": 3 if goals > goals_against else 1 if goals == goals_against else 0,
                    "goals": goals,
                    "goals_against": goals_against,
                    "goals_fh": goals_fh,
                    "goals_sh": goals_sh,
                    "goals_against_fh": goals_against_fh,
                    "goals_against_sh": goals_against_sh,
                    "matches": wins + loss + draws,
                }
            )

team_matches = (
    pd.DataFrame(team_match_rows)
    .sort_values(["season", "team", "date", "time"])
    .reset_index(drop=True)
)

teams_history: pd.DataFrame = (
    team_matches.groupby(["team", "season"], sort=False)
    .agg(
        matches=("matches", "sum"),
        wins=("wins", "sum"),
        loss=("loss", "sum"),
        draws=("draws", "sum"),
        points=("points", "sum"),
        points_per90=("points", "mean"),
        goals=("goals", "sum"),
        goals_per90=("goals", "mean"),
        goals_against=("goals_against", "sum"),
        goals_against_per90=("goals_against", "mean"),
        goals_fh=("goals_fh", "sum"),
        goals_sh=("goals_sh", "sum"),
        goals_against_fh=("goals_against_fh", "sum"),
        goals_against_sh=("goals_against_sh", "sum"),
        ts_goals=("goals", list),
        ts_goals_against=("goals_against", list),
    )
    .reset_index()
    .sort_values(["season", "team"])
    .reset_index(drop=True)
)

teams_history.head(10)

,team,season,matches,wins,loss,draws,points,points_per90,goals,goals_per90,goals_against,goals_against_per90,goals_fh,goals_sh,goals_against_fh,goals_against_sh,ts_goals,ts_goals_against
0,AC Milan,2023-24,38,22,7,9,75,1.973684,76,2.000000,49,1.289474,36,40,18,31,"[2, 4, 2, 1, 1, 3, 2, 1, 0, 2, 0, 2, 1, 3, 2, ...","[0, 1, 1, 5, 0, 1, 0, 0, 1, 2, 1, 2, 0, 1, 3, ..."
1,AC Monza,2023-24,38,11,15,12,45,1.184211,39,1.026316,51,1.342105,17,22,22,29,"[0, 2, 0, 1, 1, 0, 1, 3, 0, 1, 3, 1, 1, 1, 1, ...","[2, 0, 3, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 2, 0, ..."
2,ACF Fiorentina,2023-24,38,17,12,9,60,1.578947,61,1.605263,46,1.210526,30,31,21,25,"[4, 2, 0, 3, 2, 1, 3, 3, 0, 0, 0, 2, 0, 3, 1, ...","[1, 2, 4, 2, 0, 1, 0, 1, 2, 1, 1, 1, 1, 0, 1, ..."
3,AS Roma,2023-24,38,18,11,9,63,1.657895,65,1.710526,46,1.210526,24,41,20,26,"[2, 1, 1, 7, 1, 1, 2, 4, 1, 0, 2, 0, 3, 2, 1, ...","[2, 2, 2, 0, 1, 4, 0, 1, 0, 1, 1, 0, 1, 1, 1, ..."
4,Atalanta BC,2023-24,38,21,11,6,69,1.815789,72,1.894737,42,1.105263,35,37,21,21,"[2, 1, 3, 2, 2, 1, 0, 2, 2, 3, 1, 1, 1, 0, 3, ...","[0, 2, 0, 3, 0, 0, 0, 3, 0, 0, 2, 1, 2, 3, 2, ..."
5,Bologna FC 1909,2023-24,38,18,6,14,68,1.789474,54,1.421053,32,0.842105,26,28,18,14,"[0, 1, 2, 0, 0, 0, 3, 2, 2, 1, 1, 1, 2, 1, 2, ...","[2, 1, 1, 0, 0, 0, 0, 2, 1, 1, 0, 2, 0, 1, 1, ..."
6,Cagliari Calcio,2023-24,38,8,18,12,36,0.947368,42,1.105263,68,1.789474,12,30,29,39,"[0, 0, 1, 0, 0, 1, 0, 1, 2, 4, 2, 1, 1, 0, 2, ...","[0, 2, 2, 0, 2, 3, 3, 4, 2, 3, 1, 2, 1, 1, 1, ..."
7,Empoli FC,2023-24,38,9,20,9,36,0.947368,29,0.763158,54,1.421053,11,18,22,32,"[0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 1, 1, 3, 1, 1, ...","[1, 2, 2, 7, 1, 0, 3, 0, 0, 3, 2, 0, 4, 1, 1, ..."
8,FC Internazionale Milano,2023-24,38,29,2,7,94,2.473684,89,2.342105,22,0.578947,41,48,10,12,"[2, 2, 4, 5, 1, 1, 4, 2, 3, 1, 2, 2, 1, 3, 4, ...","[0, 0, 0, 1, 0, 2, 0, 2, 0, 0, 1, 0, 1, 0, 0, ..."
9,Frosinone Calcio,2023-24,38,8,19,11,35,0.921053,44,1.157895,69,1.815789,19,25,31,38,"[1, 2, 0, 4, 1, 1, 0, 2, 1, 3, 2, 0, 2, 1, 0, ...","[3, 1, 0, 2, 1, 1, 2, 1, 2, 4, 1, 2, 1, 3, 0, ..."


# Normalizing teams names

In [327]:
players_df: pd.DataFrame = pd.read_csv("../data/csv/notebooks_generated/serie_a_players_history.csv")

players_df = players_df[players_df["competition"].str.contains("Serie A", case=False, na=False)]
player_teams = players_df["team"].dropna().drop_duplicates().tolist()

team_aliases = {
    "FC Internazionale Milano": "Inter",
}

for player_team in player_teams:
    teams_history["team"] = teams_history["team"].apply(
        lambda history_team: player_team
        if set(player_team.casefold().split()).issubset(
            set(history_team.casefold().split())
        )
        else history_team
    )

teams_history["team"] = teams_history["team"].replace(team_aliases)
teams_history["team"]

0             Milan
1             Monza
2        Fiorentina
3              Roma
4          Atalanta
5           Bologna
6          Cagliari
7            Empoli
8             Inter
9         Frosinone
10            Genoa
11    Hellas Verona
12         Juventus
13            Lazio
14           Napoli
15           Torino
16            Lecce
17      Salernitana
18         Sassuolo
19          Udinese
20            Milan
21            Monza
22       Fiorentina
23             Roma
24         Atalanta
25          Bologna
26         Cagliari
27             Como
28           Empoli
29            Inter
30            Genoa
31    Hellas Verona
32         Juventus
33            Parma
34            Lazio
35           Napoli
36           Torino
37            Lecce
38          Udinese
39          Venezia
40            Milan
41             Pisa
42       Fiorentina
43             Roma
44         Atalanta
45          Bologna
46         Cagliari
47             Como
48            Inter
49            Genoa


## Store `teams_history.csv`

In [328]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
teams_history.to_csv(OUTPUT_PATH, index=False)

print(f"Stored {len(teams_history):,} rows in {OUTPUT_PATH.resolve()}")

Stored 60 rows in /Users/gianlucapanzani/Desktop/Fantacalcio/src/data/csv/notebooks_generated/serie_a_teams_history.csv
